# Spectral norm by power iteration with fori_loop

Spectral normalisation divides a weight matrix by its largest singular value $\sigma_{\max}$ to bound the layer's Lipschitz constant. A full SVD is too slow to run every training step, so $\sigma_{\max}$ is estimated by a few rounds of power iteration.

Write `spectral_norm(W, u0, n_iters)`:

- `W`: `(M, N)` float matrix; `u0`: `(M,)` starting vector (not zero); `n_iters`: int $\ge 1$.
- Let $\mathrm{nrm}(z) = z / (\lVert z \rVert_2 + 10^{-12})$.
- Start with $u = \mathrm{nrm}(u_0)$, then repeat `n_iters` times:
  1. $v \leftarrow \mathrm{nrm}(W^\top u)$
  2. $u \leftarrow \mathrm{nrm}(W v)$
- Finally $\sigma = u^\top W v$ using the last $u$ and $v$.

Return `(sigma, u, v)`: a float32 scalar and float32 vectors of shapes `(M,)` and `(N,)`. Run the loop with `jax.lax.fori_loop`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def spectral_norm(W, u0, n_iters):
    """Return (sigma, u (M,), v (N,)) after n_iters power-iteration rounds."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[3.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 1.0], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": 2.8635644912719727, "dtype": "float32"}, {"__nd__": [0.9938837885856628, 0.11043152958154678], "dtype": "float32"}, {"__nd__": [0.9486833810806274, 0.3162277936935425], "dtype": "float32"}]}}, {"a": [{"__nd__": [[3.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 1.0], "dtype": "float64"}, 20], "e": {"__tup__": [{"__nd__": 3.0, "dtype": "float32"}, {"__nd__": [1.0, 8.22527204619534e-20], "dtype": "float32"}, {"__nd__": [1.0, 2.467581613858602e-19], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 2.0, 0.0], [0.0, 1.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 0.0], "dtype": "float64"}, 3], "e": {"__tup__": [{"__nd__": 2.4494566917419434, "dtype": "float32"}, {"__nd__": [0.8954600095748901, 0.4451419711112976], "dtype": "float32"}, {"__nd__": [0.3676781952381134, 0.9128561615943909, 0.17749981582164764], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.3700247693240037, 0.994078692032213, 0.41585748064632017, -0.6181637087164065], [0.6721408015696635, -1.450049505885249, 0.5934846357693563, -0.5618297235440954], [0.631453096950567, 0.4390906518766011, -0.76724880591175, 0.5283687488069336], [0.3419276735141769, -0.6461430903975194, 2.001150555213166, 0.80195993578643], [-1.1820652577744168, -0.9879819159784637, 0.3168316854272728, 0.3120442520839395], [-0.7268560483413962, 1.2158778403220516, 0.09701711609506954, -0.8626833404289537]], "dtype": "float64"}, {"__nd__": [-0.3745977101286351, -0.17879339436774003, -1.5533105934432883, -0.8322121471805926, -1.809912000528707, -0.29355617314731525], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": 2.3608009815216064, "dtype": "float32"}, {"__nd__": [0.18359620869159698, -0.15366245806217194, 0.24248912930488586, -0.701066255569458, -0.5544997453689575, 0.2914029061794281], "dtype": "float32"}, {"__nd__": [0.4323098063468933, 0.4588741660118103, -0.5228378772735596, -0.57374507188797], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.3700247693240037, 0.994078692032213, 0.41585748064632017, -0.6181637087164065], [0.6721408015696635, -1.450049505885249, 0.5934846357693563, -0.5618297235440954], [0.631453096950567, 0.4390906518766011, -0.76724880591175, 0.5283687488069336], [0.3419276735141769, -0.6461430903975194, 2.001150555213166, 0.80195993578643], [-1.1820652577744168, -0.9879819159784637, 0.3168316854272728, 0.3120442520839395], [-0.7268560483413962, 1.2158778403220516, 0.09701711609506954, -0.8626833404289537]], "dtype": "float64"}, {"__nd__": [0.612639535701211, -1.2858506769982023, 0.32208625997571905, -1.1826669740084903, -0.16362570180203553, -0.5161660210815051], "dtype": "float64"}, 50], "e": {"__tup__": [{"__nd__": 2.9262571334838867, "dtype": "float32"}, {"__nd__": [0.2546147406101227, -0.4827152192592621, 0.17957285046577454, -0.6475881934165955, -0.27715814113616943, 0.4168018698692322], "dtype": "float32"}, {"__nd__": [-0.17156343162059784, 0.7623932361602783, -0.5678498148918152, -0.25859054923057556], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.212183196659453, -0.5490984987838816, -1.2254692506871665, 0.2935107219230474, 0.04617552533742384, 1.199963415268209, -1.463253289545785, 1.0927759442700036], [-0.07875734069918894, 0.4053617273724596, -0.18846678911157366, 1.5469678237589886, 1.3860177942471807, 1.0685069539798961, 0.2594567645115784, -2.3378152457678594], [0.5563281397304826, -0.03275917922956447, -0.1339695891109679, 0.43455966825756104, 1.1934595796502443, 0.26727027849946766, -0.4664258685088374, -0.7777408134163687]], "dtype": "float64"}, {"__nd__": [1.0, 1.0, 1.0], "dtype": "float64"}, 10], "e": {"__tup__": [{"__nd__": 3.632737159729004, "dtype": "float32"}, {"__nd__": [-0.1888587325811386, 0.9065553545951843, 0.3774784207344055], "dtype": "float32"}, {"__nd__": [0.09967149794101715, 0.12567375600337982, 0.00111807428766042, 0.4167090952396393, 0.46801328659057617, 0.23384545743465424, 0.09045535326004028, -0.7202233672142029], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 3.0, 4.0], [0.0, 6.0, 8.0], [0.0, 6.0, 8.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 1.0], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": 15.0, "dtype": "float32"}, {"__nd__": [0.3333333134651184, 0.6666666269302368, 0.6666666269302368], "dtype": "float32"}, {"__nd__": [0.0, 0.6000000238418579, 0.8000000715255737], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-2.0, -0.0, -0.0, -0.0], [-0.0, -2.0, -0.0, -0.0], [-0.0, -0.0, -2.0, -0.0], [-0.0, -0.0, -0.0, -2.0]], "dtype": "float64"}, {"__nd__": [1.0, -1.0, 0.5, 0.0], "dtype": "float64"}, 5], "e": {"__tup__": [{"__nd__": 2.000000238418579, "dtype": "float32"}, {"__nd__": [0.6666666865348816, -0.6666666865348816, 0.3333333432674408, 0.0], "dtype": "float32"}, {"__nd__": [-0.6666666865348816, 0.6666666865348816, -0.3333333432674408, 0.0], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(spectral_norm, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")